# Does buying the dip work? I tested 63 versions of it

**Out of Sample** — [read the article](https://arthurnoo.pythonanywhere.com/does-buying-the-dip-work)

This notebook reproduces every figure and every number in the article.

## Three ways to run it

| Where | What you need | Link |
| --- | --- | --- |
| **In your browser** | nothing at all | [Open in Colab](https://colab.research.google.com/github/Arthurnoo/out-of-sample/blob/main/notebooks/buying-the-dip.ipynb) |
| **In VS Code** | VS Code installed | [Clone and open](vscode://vscode.git/clone?url=https://github.com/Arthurnoo/out-of-sample) |
| **Anywhere else** | Python 3.10+ | you already have the file |

The next cell installs whatever is missing, so in every case you can simply
**Run All** and watch. The first run downloads prices and takes a minute or two.

## How to read it

Each section matches a section of the article. Run a cell, look at what it
prints, then read the note underneath explaining what to make of it.

**Section 0 holds every parameter you might want to change.** Everything after
it is written so that changing those values changes the whole notebook. That is
the point: the numbers below are mine, and the interesting question is what
happens to them when you move something.

Your numbers may differ slightly from the article if you run this later. The
data keeps growing, which is itself worth noticing.

## If you find a mistake

Tell me. There is a form at the bottom of the article, and a wrong number
corrected in public is worth more than a right one nobody checked.

In [ ]:
# Installs anything missing, then does nothing on later runs.
# This is what makes "Run All" work in Colab, in VS Code, or on a bare machine.
import importlib, subprocess, sys

for package in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy']:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f'installing {package} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

print('ready')

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e4e9ee', 'axes.facecolor': 'white'})

## 0. The knobs

Everything you might reasonably want to change lives in the next cell. Change a
value, run the notebook again, and every table and figure below follows.

| Knob | What it does | Try |
| --- | --- | --- |
| `SYMBOL` | the market the whole study runs on | `QQQ`, `GLD`, `BTC-USD` |
| `START` | first day of the sample | `2010-01-01` to test the recent half |
| `LOOKBACKS` | how many days back the rule looks to call it a dip | add `7`, `15` |
| `HOLDINGS` | how many days it holds after buying | add `30` |
| `COST_BPS` | cost per trade, in hundredths of a percent | `1`, `5`, `10` |
| `DIP_THRESHOLD` | how big the fall must be, as a fraction | `0.02` for 2% |
| `GO_SHORT` | also sell after a rise, not just buy after a fall | `True` |
| `N_SHUFFLES` | draws in the permutation test | lower it to go faster |

The two that change the conclusion most are `COST_BPS` and `START`. The two
most interesting to play with are `DIP_THRESHOLD` and `GO_SHORT`, because
neither is in the article and I do not know what you will find.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  PARAMETERS — this is the only cell you need to edit
# ─────────────────────────────────────────────────────────────────────

SYMBOL = 'SPY'            # any Yahoo Finance ticker
START  = '2000-01-01'     # start of the sample

LOOKBACKS = [1, 2, 3, 5, 10, 21, 63, 126, 252]   # days looked back
HOLDINGS  = [1, 2, 3, 5, 10, 21, 63]             # days held

COST_BPS      = 0.0       # cost per trade in basis points (1 bp = 0.01%)
DIP_THRESHOLD = 0.0       # 0.0 = any loss counts; 0.02 = only falls bigger than 2%
GO_SHORT      = False     # True also sells after a rise, removing market exposure
N_SHUFFLES    = 500       # permutation test draws

# A market trades about 252 days a year. Every annualised number below uses
# this, and getting it wrong is the quietest way to make a Sharpe ratio lie.
BARS_PER_YEAR = 252

## 2. The data

One function, cached in memory by `yfinance` itself. `auto_adjust=True` matters:
it adjusts for dividends and splits, and without it a stock paying a dividend
looks like it fell on the ex-date, which would create a fake dip for us to buy.

In [ ]:
def get_returns(symbol, start=START):
    h = yf.Ticker(symbol).history(start=start, auto_adjust=True)
    if h.index.tz is not None:
        h.index = h.index.tz_localize(None)
    r = h['Close'].pct_change().dropna()
    r.name = symbol
    return r

spy = get_returns('SPY')
print(f"{len(spy)} trading days, {spy.index[0].date()} to {spy.index[-1].date()}")
print(f"mean daily return: {spy.mean()*1e4:.2f} bp    daily volatility: {spy.std()*100:.2f}%")

## 3. The rule

Two parameters and no discretion.

- **L**: if the total return over the last `L` days is negative, that counts as a dip.
- **H**: having bought, hold for `H` days.

The one subtlety is overlap. A signal can fire every day, so with `H = 5` you
would hold five positions at once. Rather than ignore signals while invested, we
hold `1/H` of capital per live signal. That is the standard construction from
Jegadeesh and Titman (1993) and it uses all the data.

The consequence is that consecutive daily returns of this strategy overlap and
are **not independent**, which is why the t-statistics further down use
Newey-West standard errors instead of the naive formula.

In [ ]:
def dip_strategy(r, lookback, holding, cost_bps=COST_BPS,
                 threshold=None, go_short=None):
    threshold = DIP_THRESHOLD if threshold is None else threshold
    go_short = GO_SHORT if go_short is None else go_short

    x = r.to_numpy(dtype=float)
    n = len(x)

    # the signal: did the last `lookback` days add up to a loss bigger than
    # `threshold`? With threshold = 0 that is simply "did it fall at all".
    cum = pd.Series(x).rolling(lookback).sum().to_numpy()
    fired = (cum < -threshold).astype(float)
    if go_short:
        # also sell after a rise: the position becomes +1, 0 or -1, which
        # strips out most of the plain market exposure
        fired = fired - (cum > threshold).astype(float)
    fired[:lookback] = 0.0

    # weight = share of the last `holding` days on which a signal fired
    w = pd.Series(fired).rolling(holding).mean().fillna(0.0).to_numpy()

    # decided at today's close, earns tomorrow's return: this shift is the
    # difference between a backtest and a time machine
    pos = np.zeros(n)
    pos[1:] = w[:-1]

    turnover = np.abs(np.diff(pos, prepend=0.0))
    pnl = pos * x - turnover * cost_bps / 1e4
    return pd.Series(pnl, index=r.index), pd.Series(pos, index=r.index), turnover.sum()

pnl, pos, turn = dip_strategy(spy, lookback=2, holding=1)
print(f"invested {pos.mean()*100:.0f}% of the time, {turn/(len(spy)/252):.0f} units of turnover per year")

## 4. Measuring it

Two helpers. The Sharpe ratio is return divided by volatility, annualised. The
Newey-West t-statistic answers "could this be luck", correcting for the fact
that our returns overlap.

In [ ]:
def summarise(pnl, bars_per_year=BARS_PER_YEAR):
    x = pnl.to_numpy(dtype=float)
    eq = (1 + x).cumprod()
    dd = eq / np.maximum.accumulate(eq) - 1
    years = len(x) / bars_per_year
    sd = x.std(ddof=1)
    return {'sharpe': x.mean() / sd * np.sqrt(bars_per_year) if sd > 0 else 0.0,
            'cagr_pct': (eq[-1] ** (1 / years) - 1) * 100,
            'vol_pct': sd * np.sqrt(bars_per_year) * 100,
            'max_dd_pct': dd.min() * 100}


def newey_west_t(x, lags=None):
    """t-statistic of the mean, corrected for autocorrelation in the series.
    A naive t assumes independent observations; overlapping holding periods
    break that assumption and inflate the t by a large factor."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    if lags is None:
        lags = int(np.floor(4 * (n / 100) ** (2 / 9)))
    xc = x - x.mean()
    s = float(xc @ xc) / n
    for l in range(1, lags + 1):
        s += 2 * (1 - l / (lags + 1)) * float(xc[l:] @ xc[:-l]) / n
    return float(x.mean() / np.sqrt(max(s, 1e-300) / n))

bh = summarise(spy)
print(f"Buy and hold:  Sharpe {bh['sharpe']:.2f}   CAGR {bh['cagr_pct']:.2f}%   worst fall {bh['max_dd_pct']:.1f}%")
print(f"Dip rule L=2 H=1:  Sharpe {summarise(pnl)['sharpe']:.2f}   t = {newey_west_t(pnl.to_numpy()):.2f}")

## 5. The grid

This is the figure that carries the article. Sixty-three versions of the same
idea, one per cell.

What to look for is not the best number. It is whether the good cells form a
**connected region**. A single bright cell surrounded by dull ones is noise; a
smooth gradient means the effect degrades gracefully as you change the
parameter, which is what a real effect does.

In [ ]:
LOOKBACKS = [1, 2, 3, 5, 10, 21, 63, 126, 252]
HOLDINGS  = [1, 2, 3, 5, 10, 21, 63]

def build_grid(r, cost_bps=0.0):
    rows = []
    for L in LOOKBACKS:
        for H in HOLDINGS:
            p, pos, _ = dip_strategy(r, L, H, cost_bps)
            m = summarise(p)
            rows.append({'lookback': L, 'holding': H, 'sharpe': m['sharpe'],
                         't_nw': newey_west_t(p.to_numpy()),
                         'exposure': pos.mean()})
    return pd.DataFrame(rows)

grid = build_grid(spy)
best = grid.loc[grid.sharpe.idxmax()]
print(f"Best: look back {int(best.lookback)}d, hold {int(best.holding)}d -> "
      f"Sharpe {best.sharpe:.2f}, t = {best.t_nw:.2f}")
print(f"Cells beating buy and hold: {(grid.sharpe > bh['sharpe']).sum()} of {len(grid)}")
print(f"Cells with t above 2:       {(grid.t_nw > 2).sum()} of {len(grid)}")

In [ ]:
def show_grid(df, value='sharpe', title=''):
    piv = df.pivot(index='lookback', columns='holding', values=value)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    lim = np.nanmax(np.abs(piv.values))
    im = ax.imshow(piv.values, cmap='RdYlGn', vmin=-lim, vmax=lim, aspect='auto')
    ax.set_xticks(range(len(piv.columns)), piv.columns)
    ax.set_yticks(range(len(piv.index)), piv.index)
    ax.set_xlabel('days held'); ax.set_ylabel('days looked back')
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f'{piv.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
    ax.grid(False); ax.set_title(title, loc='left')
    fig.colorbar(im, ax=ax, fraction=0.035)
    plt.show()

show_grid(grid, 'sharpe', f"Sharpe by (lookback, holding). Buy and hold = {bh['sharpe']:.2f}")

The top rows are green, the bottom rows are not. Looking back one to five days
works; looking back six months or a year does not.

Note what that means for the advice as people actually use it. Nobody says "buy
the dip" about two red days. They mean a real drop, weeks or months. That is the
bottom of the grid, and it is the part that does not work.

In [ ]:
show_grid(grid, 't_nw', 'Newey-West t-statistic. Above 2 is the usual bar.')

## 6. Does it travel?

A result that exists on one market is usually a fact about that market's
history rather than about markets. Run the same grid everywhere.

In [ ]:
UNIVERSE = {'SPY': 'S&P 500', 'QQQ': 'Nasdaq 100', 'IWM': 'US small caps',
            'EFA': 'Developed ex-US', 'EEM': 'Emerging markets', 'GLD': 'Gold',
            'TLT': 'US Treasuries 20y', 'USO': 'Oil', 'AAPL': 'Apple',
            'KO': 'Coca-Cola', 'XOM': 'Exxon'}

rows = []
for sym, name in UNIVERSE.items():
    try:
        r = get_returns(sym)
        if len(r) < 1500:
            continue
        g = build_grid(r)
        b = g.loc[g.sharpe.idxmax()]
        rows.append({'asset': name, 'buy_hold': summarise(r)['sharpe'],
                     'best_dip': b.sharpe, 'L': int(b.lookback), 'H': int(b.holding),
                     't': b.t_nw, 'cells_t_above_2': int((g.t_nw > 2).sum())})
    except Exception as e:
        print(f'{name}: {e}')

cross = pd.DataFrame(rows).sort_values('best_dip', ascending=False)
print(cross.to_string(index=False, float_format=lambda v: f'{v:7.2f}'))

Ten of eleven markets agree, and in every case the winning parameters are
short. Oil is the exception and fails completely: its best cell has a
t-statistic below 1 and not one of its sixty-three combinations clears the bar.
Oil trends rather than reverts, so the honest reading is that this is a result
about equity-like assets, not about markets in general.

## 7. The three tests that decide whether to believe it

### a) Am I using the future?

This is the error that quietly inflates most amateur backtests, and it is worth
checking rather than assuming. Today's position should be explained by
**yesterday's** return, not today's.

In [ ]:
pnl, pos, _ = dip_strategy(spy, 2, 1)
same_day = pd.Series(pos.values).corr(pd.Series(spy.values))
prev_day = pd.Series(pos.values[1:]).corr(pd.Series(spy.values[:-1]))
print(f"correlation with yesterday's return : {prev_day:+.3f}   (should be strong: that IS the signal)")
print(f"correlation with today's return     : {same_day:+.3f}   (should be near zero)")
print(f"first position is flat              : {pos.iloc[0] == 0}")

### b) How much is skill and how much is just being invested?

Shuffle the order of the days. That destroys any day-to-day pattern while
keeping the distribution of returns identical. Then run the strategy again on
the shuffled series, many times.

Whatever Sharpe survives the shuffle was never about dips. It is the reward for
holding a market that drifts upward.

In [ ]:
rng = np.random.default_rng(0)
real = summarise(dip_strategy(spy, 2, 1)[0])['sharpe']
null = np.empty(500)
for i in range(500):
    shuffled = pd.Series(rng.permutation(spy.to_numpy()), index=spy.index)
    null[i] = summarise(dip_strategy(shuffled, 2, 1)[0])['sharpe']

p_value = float((np.abs(null) >= abs(real)).mean())
print(f"real Sharpe          : {real:.3f}")
print(f"average when shuffled: {null.mean():.3f}   <- this part is NOT timing")
print(f"attributable to timing: {real - null.mean():.3f}")
print(f"percentile of the real result: {(null < real).mean()*100:.1f}    p-value: {p_value:.4f}")

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.hist(null, bins=40, color='#8b98a6', alpha=.7, edgecolor='white')
ax.axvline(real, color='#0e8f5f', lw=2)
ax.set_xlabel('Sharpe when the order of days is destroyed'); ax.set_ylabel('draws')
ax.set_title('Permutation test', loc='left')
plt.show()

Half the headline number is simply the market going up. The timing part is real
(the p-value is small) but it is half of what the raw Sharpe suggested.

Ask this of any backtest you ever read: **what would this earn if the pattern it
claims to exploit did not exist?**

### c) What does it cost?

This is where the strategy ends. The best cell turns over about eighty-four
times a year, and each round trip costs something.

In [ ]:
rows = []
for c in [0, 0.5, 1, 1.5, 2, 3, 4, 5, 10, 20]:
    p, _, _ = dip_strategy(spy, 2, 1, cost_bps=c)
    rows.append({'cost_bps': c, 'sharpe': summarise(p)['sharpe']})
costs = pd.DataFrame(rows)
costs['beats_buy_and_hold'] = costs.sharpe > bh['sharpe']
print(f"buy and hold sits at {bh['sharpe']:.3f}\n")
print(costs.to_string(index=False, float_format=lambda v: f'{v:8.3f}'))

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.plot(costs.cost_bps, costs.sharpe, marker='o', color='#2d6fa8')
ax.axhline(bh['sharpe'], color='#c83c50', ls='--', label='buy and hold')
ax.set_xlabel('cost per trade (basis points)'); ax.set_ylabel('Sharpe'); ax.legend()
ax.set_title('Where the edge ends', loc='left')
plt.show()

Somewhere between three and four basis points, dip buying stops being better
than doing nothing at all. Four basis points is four hundredths of one percent.

That is the conclusion of the piece. A strategy that trades eighty-four times a
year and has an edge of four basis points does not have an edge, it has a bill.

## 8. Things worth trying yourself

The point of shipping this file is that you can disagree with me. Some changes
that would genuinely test the result:

1. **Move `START` to 2010.** Does the grid still look the same? The article
   claims it holds in both halves of the sample; check it.
2. **Use a threshold instead of a sign.** Right now any loss counts as a dip.
   Try requiring a fall of more than 2%, or more than one standard deviation.
   Does demanding a bigger dip help, or does it just reduce the sample?
3. **Add a short leg.** Sell after a rise as well as buying after a fall. That
   removes most of the market exposure. Does the permutation test null move to
   zero, and does the Sharpe survive?
4. **Try a market where you expect it to fail.** Bitcoin, or a single small-cap.
   A result you cannot break is a result you have not tested hard enough.

If you find something I got wrong, the article has a form at the bottom. I would
rather be corrected in public than be quietly wrong.